In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Any, Optional
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

from evals.models import DeepEvalObservation, RougeObservation, ResourceObservation
from evals import Dataset

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Fetch results

In [ ]:
class CollectMetrics:
    def __init__(self):
        self.sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _extract_run_type(self, path):
        run = path.split("/")[-1].replace("llm-as-judge_", "")
        return "_".join(run.split("_")[:-1])

    def organize_runs_by_type(self, eval_res):
        runs = {}
        for path in eval_res.keys():
            ext_path = self._extract_run_type(path)
            if ext_path in runs:
                runs[ext_path].append(eval_res[path])
            else:
                runs[ext_path] = [eval_res[path]]
        return runs

    def _extract_query_metrics(self, query):
        correctness = relevancy = completeness = 0.0
        for metric in query.metrics_data:
            if metric.name == 'correctness [GEval]':
                correctness = metric.score
            elif metric.name == 'relevancy':
                relevancy = metric.score
            elif metric.name == 'completeness [GEval]':
                completeness = metric.score
        return {"correctness": correctness, "relevancy": relevancy, "completeness": completeness}

    def self_consistency_score(self, outputs: list[str]) -> float:
        """Cosine similarity between sentence embeddings of multiple outputs for the same query."""
        if len(outputs) <= 1:
            return 1.0
        embeddings = self.sentence_transformer.encode(outputs)
        sim_matrix = cosine_similarity(embeddings)
        n = len(outputs)
        mask = np.triu(np.ones((n, n), dtype=bool), k=1)
        return float(sim_matrix[mask].mean())

    def _rouge_l(self, expected_output, actual_output):
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
        return scorer.score(target=expected_output, prediction=actual_output)

    # ── Public collection methods ─────────────────────────────────────────────

    def collect_deepeval_metrics(self, runs) -> list[DeepEvalObservation]:
        """One DeepEvalObservation per query × eval_run from LLM-as-judge."""
        observations = []
        for run in runs:
            for res in run.results:
                for query in res.test_results:
                    m = self._extract_query_metrics(query)
                    observations.append(DeepEvalObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=query.additional_metadata.get("query_id"),
                        session_id=query.additional_metadata.get("session_id"),
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        name=query.name,
                        correctness=m["correctness"],
                        relevancy=m["relevancy"],
                        completeness=m["completeness"],
                        success=query.success,
                    ))
        return observations

    def collect_reference_metrics(self, runs) -> list[RougeObservation]:
        """One RougeObservation per query × eval_run for reference-based metrics."""
        observations = []
        for run in runs:
            for res in run.results:
                for query in res.test_results:
                    score = self._rouge_l(query.expected_output, query.actual_output)
                    observations.append(RougeObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=query.additional_metadata.get("query_id"),
                        session_id=query.additional_metadata.get("session_id"),
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        name=query.name,
                        rouge_precision=score['rougeL'].precision,
                        rouge_recall=score['rougeL'].recall,
                        rouge_fmeasure=score['rougeL'].fmeasure,
                        actual_output=query.actual_output,
                    ))
        return observations

    def collect_resource_metrics(self, runs) -> list[ResourceObservation]:
        """One ResourceObservation per query × eval_run with per-query token counts and duration."""
        observations = []
        for run in runs:
            per_query = (run.token_counts or {}).get("per_query", {})
            for res in run.results:
                for query in res.test_results:
                    qid = query.additional_metadata.get("query_id") or "unknown"
                    q_tokens = per_query.get(qid, {})
                    observations.append(ResourceObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=qid,
                        session_id=query.additional_metadata.get("session_id"),
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        input_tokens=q_tokens.get("input_tokens", 0),
                        output_tokens=q_tokens.get("output_tokens", 0),
                        total_tokens=q_tokens.get("total_tokens", 0),
                        llm_calls=q_tokens.get("llm_calls", 0),
                        duration=q_tokens.get("duration", 0.0),
                    ))
        return observations

## Evaluation Analysis

In [3]:
ds = Dataset("test")
cm = CollectMetrics()
eval_res = ds.load_evaluation_results()
all_runs = cm.organize_runs_by_type(eval_res)
all_runs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'google_gemini-2.5-flash_baseline': [EvalOutput(dataset_name='test', project_id='b3f45644-6222-4593-8955-cf4d8e0b00d5', user_id='53d63d18-cfa1-416e-96e8-770c8f66507b', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', llm_model='google_gemini-2.5-flash', agent_type='baseline', created_at='2026-02-27T15:31:23.504609', results=[EvaluationResult(test_results=[TestResult(name='Turn 1 in session Prosjekt-initialisering', success=True, metrics_data=[MetricData(name='correctness [GEval]', threshold=0.5, success=True, score=0.9018833128651359, reason="The actual output closely aligns with the expected output, accurately summarizing the key events: the purchase and overtakelse dates, the seller's notification of a leak before overtakelse, the recurrence of the leak after overtakelse, the findings of the skaderapporter regarding the membrane puncture, and the technical report in March 2020 highlighting construction issues. The output also correctly identifies the core issue as misrepresentati

In [4]:
deepeval_obs = []
for runs in all_runs.values():
    deepeval_obs.extend(cm.collect_deepeval_metrics(runs))

deepeval_obs[:2], len(deepeval_obs)

([DeepEvalObservation(dataset_name='test', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', query_id='f3f226bb-eb5c-4b87-8d96-dfa425614f42', session_id=None, llm_model='google_gemini-2.5-flash', agent_type='baseline', name='Turn 1 in session Prosjekt-initialisering', correctness=0.9018833128651359, relevancy=0.0, completeness=0.9128048583516355, success=True),
  DeepEvalObservation(dataset_name='test', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', query_id='070f570b-f9d0-454f-90ca-84f66ede1a31', session_id=None, llm_model='google_gemini-2.5-flash', agent_type='baseline', name='Turn 2 in session Prosjekt-initialisering', correctness=0.8083314984233272, relevancy=0.0, completeness=0.7432988709227093, success=True)],
 35)

In [5]:
rouge_obs = []
for runs in all_runs.values():
    rouge_obs.extend(cm.collect_reference_metrics(runs))

rouge_obs[:2], len(rouge_obs)

([RougeObservation(dataset_name='test', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', query_id='f3f226bb-eb5c-4b87-8d96-dfa425614f42', session_id=None, llm_model='google_gemini-2.5-flash', agent_type='baseline', name='Turn 1 in session Prosjekt-initialisering', rouge_precision=0.13428571428571429, rouge_recall=0.2088888888888889, rouge_fmeasure=0.1634782608695652, actual_output='Kjernen i saken er at kjøperne, Anders og Berit Kristiansen, oppdaget alvorlige mangler ved eiendommen Fjellveien 42A, spesielt knyttet til utleiedelen, kort tid etter overtakelsen 1. august 2019. Dette står i kontrast til selger Carl Danielsens forsikringer før kjøpet om at alt var i orden, ingen fuktproblemer forelå, og at alle byggetillatelser og bruksendringer var godkjent.\n\n**Faktisk bakgrunn og utvikling:**\n\n*   **Før kjøp (mai 2019):** Kjøperne spurte spesifikt om utleiedelen var godkjent, byggetillatelser og ferdigattester, samt fuktproblemer. Selger svarte bekreftende på at utleiedelen var go

In [7]:
resource_obs = []
for runs in all_runs.values():
    resource_obs.extend(cm.collect_resource_metrics(runs))

resource_obs[:2], len(resource_obs)

([ResourceObservation(dataset_name='test', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', query_id='f3f226bb-eb5c-4b87-8d96-dfa425614f42', session_id=None, llm_model='google_gemini-2.5-flash', agent_type='baseline', input_tokens=64870, output_tokens=13622, total_tokens=78492, llm_calls=9, duration=0.0),
  ResourceObservation(dataset_name='test', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', query_id='070f570b-f9d0-454f-90ca-84f66ede1a31', session_id=None, llm_model='google_gemini-2.5-flash', agent_type='baseline', input_tokens=64870, output_tokens=13622, total_tokens=78492, llm_calls=9, duration=0.0)],
 35)

## Statistical Modeling

In [8]:
import numpy as np
from scipy import stats 
import statsmodels.api as sm
from statsmodels.stats.power import tt_ind_solve_power  

In [9]:
class StatisticalMetrics:
    def __init__(self, significance_level: float = 0.05):
        self.significance_level = significance_level

    def check_normality(self, observations: list[DeepEvalObservation], metric: str = "correctness"):
        for agent in ["baseline", "baseline_rag", "custom"]:
            values = [getattr(obs, metric) for obs in observations if obs.agent_type == agent]
            if not values:
                continue
            result = stats.shapiro(values)
            status = "✅" if result.pvalue > self.significance_level else "❌"
            print(f"{status} {agent}: {result}")

    def t_test(self, custom_eval: list[float] | np.ndarray, comparison_eval: list[float] | np.ndarray):
        t_stat, p = stats.ttest_ind(custom_eval, comparison_eval, equal_var=False, alternative='greater')
        print(f"Custom vs. Comparison: t={t_stat:.2f}, p={p:.4f}")
        if p < self.significance_level:
            print("Custom slår Comparison (p<0.05).")

## One-sided Hypothesis Tests

**For quality/performance metrics** (correctness, relevancy, completeness, passrate):

$$
H_0: \mu_{\text{custom}} \leq \mu_{\text{baseline}} \qquad H_1: \mu_{\text{custom}} > \mu_{\text{baseline}}
$$

(and analogously vs. baseline+RAG)

**For cost/efficiency metrics** (token usage, inference time):

$$
H_0: \mu_{\text{custom}} \geq \mu_{\text{baseline}} \qquad H_1: \mu_{\text{custom}} < \mu_{\text{baseline}}
$$

(and analogously vs. baseline+RAG)

**Notation:**

- $\mu_{\text{custom}}$ = population mean for the custom agent  
- $\mu_{\text{baseline}}$ = population mean for the baseline agent  
- $\mu_{\text{baseline+RAG}}$ = population mean for baseline with RAG  

All tests are one-tailed / one-sided.

In [9]:
from collections import Counter
Counter(obs.agent_type for obs in deepeval_obs)

Counter({'baseline': 25, 'baseline_rag': 5, 'custom': 5})

## EDA
### Exploratory data analysis

In [10]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

In [11]:
df_deepeval  = pd.DataFrame(obs.model_dump() for obs in deepeval_obs)
df_rouge     = pd.DataFrame(obs.model_dump() for obs in rouge_obs)
df_resource  = pd.DataFrame(obs.model_dump() for obs in resource_obs)

# self-consistency: computed per query across eval_runs
sc = df_rouge.groupby(["query_id", "agent_type"])["actual_output"].apply(
    lambda outputs: cm.self_consistency_score(outputs.tolist())
).reset_index(name="self_consistency")
df_rouge = df_rouge.merge(sc, on=["query_id", "agent_type"], how="left")

df_deepeval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   dataset_name  35 non-null     object 
 1   eval_run_id   35 non-null     object 
 2   query_id      35 non-null     object 
 3   session_id    0 non-null      object 
 4   llm_model     35 non-null     object 
 5   agent_type    35 non-null     object 
 6   name          35 non-null     object 
 7   correctness   35 non-null     float64
 8   relevancy     35 non-null     float64
 9   completeness  35 non-null     float64
 10  success       35 non-null     bool   
dtypes: bool(1), float64(3), object(7)
memory usage: 2.9+ KB


In [12]:
px.histogram(data_frame=df_deepeval, x="correctness", color="agent_type", nbins=100, title="Correctness Distribution by Agent Type")

In [13]:
px.histogram(data_frame=df_resource, x="total_tokens", color="agent_type", nbins=20, title="Total Tokens Used by Agent Type")

In [14]:
df_resource

,dataset_name,eval_run_id,query_id,session_id,llm_model,agent_type,input_tokens,output_tokens,total_tokens,llm_calls,duration
0,test,2161608a-ae80-4fb7-9f2e-33de91e0316d,f3f226bb-eb5c-4b87-8d96-dfa425614f42,None,google_gemini-2.5-flash,baseline,64870,13622,78492,9,0.000000
1,test,2161608a-ae80-4fb7-9f2e-33de91e0316d,070f570b-f9d0-454f-90ca-84f66ede1a31,None,google_gemini-2.5-flash,baseline,64870,13622,78492,9,0.000000
2,test,2161608a-ae80-4fb7-9f2e-33de91e0316d,335b247f-0a2b-4eef-85f8-5cc0e2c6a2e2,None,google_gemini-2.5-flash,baseline,64870,13622,78492,9,0.000000
3,test,2161608a-ae80-4fb7-9f2e-33de91e0316d,82d6385b-3f42-4c89-8df6-82cea426f45a,None,google_gemini-2.5-flash,baseline,64870,13622,78492,9,0.000000
4,test,2161608a-ae80-4fb7-9f2e-33de91e0316d,b309a1f0-2399-461a-9739-047bf840516b,None,google_gemini-2.5-flash,baseline,64870,13622,78492,9,0.000000
5,test,510f7642-9b65-415f-ac5e-fcd582768b01,f3f226bb-eb5c-4b87-8d96-dfa425614f42,None,google_gemini-2.5-flash,baseline,67218,13235,80453,10,0.000000
6,test,510f7642-9b65-415f-ac5e-fcd582768b01,070f570b-f9d0-454f-90ca-84f66ede1a31,None,google_gemini-2.5-flash,baseline,67218,13235,80453,10,0.000000
7,test,510f7642-9b65-415f-ac5e-fcd582768b01,335b247f-0a2b-4eef-85f8-5cc0e2c6a2e2,None,google_gemini-2.5-flash,baseline,67218,13235,80453,10,0.000000
8,test,510f7642-9b65-415f-ac5e-fcd582768b01,82d6385b-3f42-4c89-8df6-82cea426f45a,None,google_gemini-2.5-flash,baseline,67218,13235,80453,10,0.000000
9,test,510f7642-9b65-415f-ac5e-fcd582768b01,b309a1f0-2399-461a-9739-047bf840516b,None,google_gemini-2.5-flash,baseline,67218,13235,80453,10,0.000000
